In [0]:
from pyspark.sql.functions import *
CATALOG = "capstone_8hospital"
SCHEMA = "bronze_schema"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [0]:
ENCOUNTER_TABLE = CATALOG + "." + SCHEMA + ".bronze_patient_encounters"
LAB_TABLE = CATALOG + "." + SCHEMA + ".bronze_lab_test_events"
TRANSFER_TABLE = CATALOG + "." + SCHEMA + ".bronze_bed_transfer_events"

PATIENT_TABLE = CATALOG + "." + SCHEMA + ".bronze_patients"
DEPARTMENT_TABLE = CATALOG + "." + SCHEMA + ".bronze_departments"
BED_TABLE = CATALOG + "." + SCHEMA + ".bronze_beds"

encounters = spark.table(ENCOUNTER_TABLE)
lab_events = spark.table(LAB_TABLE)
transfer_events = spark.table(TRANSFER_TABLE)

encounters.printSchema()
lab_events.printSchema()
transfer_events.printSchema()


In [0]:
print("Patient Encounters:", encounters.count())
print("Lab Test Events:", lab_events.count())
print("Bed Transfer Events:", transfer_events.count())

print("Patient Master:", spark.table(PATIENT_TABLE).count())
print("Department Master:", spark.table(DEPARTMENT_TABLE).count())
print("Bed Master:", spark.table(BED_TABLE).count())


In [0]:
total_encounters = encounters.count()
distinct_encounters = encounters.select("encounter_id").distinct().count()

duplicate_encounters = (
    encounters.groupBy("encounter_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

null_encounters = encounters.filter(col("encounter_id").isNull()).count()

display(spark.createDataFrame([
    ("Total Encounter Rows", total_encounters),
    ("Distinct Encounter IDs", distinct_encounters),
    ("Duplicate Encounter IDs", duplicate_encounters),
    ("Null Encounter IDs", null_encounters)
], ["Profiling Metric", "Count"]))

In [0]:
print("Total Lab Event Rows:", lab_events.count())
print("Distinct Lab Event IDs:", lab_events.select("lab_event_id").distinct().count())
print("Null Lab Event IDs:", lab_events.filter(col("lab_event_id").isNull()).count())


In [0]:
print("Total Transfer Rows:", transfer_events.count())
print("Distinct Transfer IDs:", transfer_events.select("transfer_id").distinct().count())
print("Null Transfer IDs:", transfer_events.filter(col("transfer_id").isNull()).count())


In [0]:
for df, col_name in [
    (encounters, "encounter_type"),
    (encounters, "triage_priority"),
    (encounters, "encounter_status"),
    (lab_events, "test_code"),
    (lab_events, "result_status"),
    (transfer_events, "transfer_reason"),
    (transfer_events, "transfer_status"),
]:
    display(df.groupBy(col_name).count().orderBy(desc("count")))


In [0]:
display(encounters.groupBy("_source_file").count().orderBy("_source_file"))

In [0]:
print("Bronze Profiling Summary")
summary_data = [
    ("Patient Encounters", total_encounters, distinct_encounters, duplicate_encounters),
    ("Lab Test Events", lab_events.count(), lab_events.select("lab_event_id").distinct().count(), lab_events.groupBy("lab_event_id").count().filter(col("count") > 1).count()),
    ("Bed Transfer Events", transfer_events.count(), transfer_events.select("transfer_id").distinct().count(), transfer_events.groupBy("transfer_id").count().filter(col("count") > 1).count()),
    ("Patient Master", spark.table(PATIENT_TABLE).count(), spark.table(PATIENT_TABLE).select("patient_id").distinct().count(), spark.table(PATIENT_TABLE).groupBy("patient_id").count().filter(col("count") > 1).count()),
    ("Department Master", spark.table(DEPARTMENT_TABLE).count(), spark.table(DEPARTMENT_TABLE).select("department_id").distinct().count(), spark.table(DEPARTMENT_TABLE).groupBy("department_id").count().filter(col("count") > 1).count()),
    ("Bed Master", spark.table(BED_TABLE).count(), spark.table(BED_TABLE).select("bed_id").distinct().count(), spark.table(BED_TABLE).groupBy("bed_id").count().filter(col("count") > 1).count())
]

df_summary = spark.createDataFrame(summary_data, ["Table", "Total", "Distinct", "Duplicates"])
display(df_summary)